In [7]:
import pandas as pd

# 1. Carrega os arquivos (que vêm com uma coluna única e gigante)
df_clientes_raw = pd.read_excel('clientes.csv')
df_acessos_raw = pd.read_excel('acessos.csv')
df_pedidos_raw = pd.read_excel('pedidos.csv')

# 2. Função mágica para pegar a primeira coluna de texto e quebrar nas vírgulas
df_clientes = df_clientes_raw.iloc[:, 0].str.split(',', expand=True)
df_clientes.columns = df_clientes_raw.columns[0].split(',')

df_acessos = df_acessos_raw.iloc[:, 0].str.split(',', expand=True)
df_acessos.columns = df_acessos_raw.columns[0].split(',')

df_pedidos = df_pedidos_raw.iloc[:, 0].str.split(',', expand=True)
df_pedidos.columns = df_pedidos_raw.columns[0].split(',')

# --- AUDITORIA FINAL DE VERIFICAÇÃO ---
print("====== AGORA SIM! COLUNAS SEPARADAS ======")
print("Clientes:", list(df_clientes.columns))
print("Acessos:", list(df_acessos.columns))
print("Pedidos:", list(df_pedidos.columns))

====== AGORA SIM! COLUNAS SEPARADAS ======
Clientes: ['ID_Cliente', 'Nome', 'Email', 'Telefone', 'Data_Nascimento']
Acessos: ['ID_Sessao', 'Nome_Cliente', 'Horario_Inicio', 'Horario_Termino', 'Valor_Carrinho', 'Compra_Finalizada']
Pedidos: ['ID_Pedido', 'ID_Sessao', 'Produto', 'Categoria', 'Quantidade', 'Preco_Unitario', 'Data_Compra']


In [8]:
# 1. Unir Clientes com Acessos usando o Nome de cada um
df_master = pd.merge(df_clientes, df_acessos, left_on='Nome', right_on='Nome_Cliente', how='inner')

# 2. Unir o resultado anterior com os Pedidos usando o identificador da sessão (ID_Sessao)
df_consolidado = pd.merge(df_master, df_pedidos, on='ID_Sessao', how='left')

# --- AUDITORIA DA BASE UNIFICADA ---
print("====== PIPELINE INTEGRADO COM SUCESSO ======")
print(f"Total de linhas na base final unificada: {df_consolidado.shape[0]}")
print(f"Total de colunas disponíveis para a OmniStore: {df_consolidado.shape[1]}\n")

# Visualizar as primeiras linhas da super tabela organizada
df_consolidado.head()

====== PIPELINE INTEGRADO COM SUCESSO ======
Total de linhas na base final unificada: 483
Total de colunas disponíveis para a OmniStore: 17



,ID_Cliente,Nome,Email,Telefone,Data_Nascimento,ID_Sessao,Nome_Cliente,Horario_Inicio,Horario_Termino,Valor_Carrinho,Compra_Finalizada,ID_Pedido,Produto,Categoria,Quantidade,Preco_Unitario,Data_Compra
0,1,João Santos,joaosantos@email.com,(11) 98765-4321,1990-05-15,S001,João Santos,09:00:15,09:05:30,150.50,Sim,P001,Smartphone,Eletronicos,1,150.50,2025-01-15
1,1,João Santos,joaosantos@email.com,(11) 98765-4321,1990-05-15,S001,João Santos,09:00:15,09:05:30,150.50,Sim,P058,Smartphone,Eletronicos,1,150.50,2025-01-15
2,1,João Santos,joaosantos@email.com,(11) 98765-4321,1990-05-15,S024,João Santos,13:30:00,13:35:00,170.00,Sim,P015,Smartphone,Eletronicos,1,170.00,2025-01-27
3,1,João Santos,joaosantos@email.com,(11) 98765-4321,1990-05-15,S024,João Santos,13:30:00,13:35:00,170.00,Sim,P072,Smartphone,Eletronicos,1,170.00,2025-01-27
4,1,João Santos,joaosantos@email.com,(11) 98765-4321,1990-05-15,S043,João Santos,17:00:00,17:05:00,95.00,Sim,P026,Smartphone,Eletronicos,1,95.00,2025-02-07


In [9]:
# 1. Converter quantidades e preços para formatos numéricos
df_consolidado['Quantidade'] = pd.to_numeric(df_consolidado['Quantidade'], errors='coerce')
df_consolidado['Preco_Unitario'] = pd.to_numeric(df_consolidado['Preco_Unitario'], errors='coerce')
df_consolidado['Valor_Carrinho'] = pd.to_numeric(df_consolidado['Valor_Carrinho'], errors='coerce')

# 2. Calcular o Valor Total de cada item comprado (Quantidade * Preço Unitário)
df_consolidado['Valor_Total_Item'] = df_consolidado['Quantidade'] * df_consolidado['Preco_Unitario']

# 3. Tratar valores nulos (Se não comprou, a quantidade e o valor viram 0)
df_consolidado['Quantidade'] = df_consolidado['Quantidade'].fillna(0)
df_consolidado['Valor_Total_Item'] = df_consolidado['Valor_Total_Item'].fillna(0)

print("====== DADOS TRATADOS E FORMATADOS ======")
print("Tipos das colunas numéricas ajustados com sucesso!")

====== DADOS TRATADOS E FORMATADOS ======
Tipos das colunas numéricas ajustados com sucesso!


In [10]:
import numpy as np

print("====== LEVANTAMENTO DE KPIs - OMNISTORE ======\n")

# KPI 1: Faturamento Total Bruto
faturamento_total = df_consolidado['Valor_Total_Item'].sum()
print(f"1. Faturamento Total da Loja: R$ {faturamento_total:,.2f}")

# KPI 2: Ticket Médio dos Pedidos (Média do valor gasto por transação real)
# Filtramos apenas linhas onde houve uma compra de fato (Preco_Unitario maior que zero)
pedidos_reais = df_consolidado[df_consolidado['Preco_Unitario'] > 0]
ticket_medio = pedidos_reais['Valor_Total_Item'].mean()
print(f"2. Ticket Médio por Item Comprado: R$ {ticket_medio:,.2f}")

# KPI 3: Taxa de Conversão de Vendas
# Quantas sessões únicas resultaram em compra de fato?
total_sessoes = df_consolidado['ID_Sessao'].nunique()
# No seu dataset, a coluna 'Compra_Finalizada' indica o status (vamos garantir que pegamos as compras feitas)
sessoes_com_compra = df_consolidado[df_consolidado['ID_Pedido'].notna()]['ID_Sessao'].nunique()
taxa_conversao = (sessoes_com_compra / total_sessoes) * 100
print(f"3. Taxa de Conversão de Clientes: {taxa_conversao:.2f}%")

# KPI 4: Categoria de Produto Mais Vendida (em quantidade)
categoria_mais_vendida = pedidos_reais.groupby('Categoria')['Quantidade'].sum().idxmax()
qtd_categoria_mais_vendida = pedidos_reais.groupby('Categoria')['Quantidade'].sum().max()
print(f"4. Categoria Campeã de Vendas: {categoria_mais_vendida} ({int(qtd_categoria_mais_vendida)} itens vendidos)")

# KPI 5: Produto Líder de Faturamento
produto_lider = pedidos_reais.groupby('Produto')['Valor_Total_Item'].sum().idxmax()
faturamento_produto_lider = pedidos_reais.groupby('Produto')['Valor_Total_Item'].sum().max()
print(f"5. Produto Líder em Faturamento: {produto_lider} (R$ {faturamento_produto_lider:,.2f})")

====== LEVANTAMENTO DE KPIs - OMNISTORE ======

1. Faturamento Total da Loja: R$ 52,896.00
2. Ticket Médio por Item Comprado: R$ 154.67
3. Taxa de Conversão de Clientes: 53.92%
4. Categoria Campeã de Vendas: Eletronicos (142 itens vendidos)
5. Produto Líder em Faturamento: Smartphone (R$ 6,926.00)


In [11]:
# Exportando a base consolidada para um arquivo Excel real
df_consolidado.to_excel('Resultado_Final_OmniStore.xlsx', index=False)
print("Arquivo 'Resultado_Final_OmniStore.xlsx' gerado com sucesso na barra lateral!")

Arquivo 'Resultado_Final_OmniStore.xlsx' gerado com sucesso na barra lateral!
